In [ ]:
# CONFIG — All Drive paths in one place. Run this cell first, then edit here only.
DRIVE_ROOT      = "/content/drive/MyDrive/LostArchive"
TRAIN_DATA_DIR  = f"{DRIVE_ROOT}/prod_train/img"
OUTPUT_DIR_LORA = f"{DRIVE_ROOT}/prod_train/output"
LOG_DIR         = f"{DRIVE_ROOT}/prod_train/log"
BASE_MODEL      = f"{DRIVE_ROOT}/models/realisticVisionV60B1_v51VAE.safetensors"
CHECKPOINT      = f"{OUTPUT_DIR_LORA}/BushraStyle_prod_v1-step00004000.safetensors"

In [ ]:
# CELL 1 — Mount Drive + Check GPU
# Run this first every session. Must show T4 with ~15360MiB VRAM.
# If no GPU: Runtime > Disconnect and delete runtime > Reconnect

from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import sys
import subprocess

print("Python version:", sys.version)

# Check what torch is already installed
result = subprocess.run([sys.executable, "-m", "pip", "show", "torch"],
                       capture_output=True, text=True)
print("Current torch:", result.stdout)

# Check available disk space
result2 = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print("Disk space:", result2.stdout)

# Try to find torch 2.4.1 wheels for this python version
result3 = subprocess.run([sys.executable, "-m", "pip", "index", "versions", "torch",
                         "--index-url", "https://download.pytorch.org/whl/cu118"],
                        capture_output=True, text=True)
print("Available torch versions:", result3.stdout[:500])
print("Errors:", result3.stderr[:500])

In [ ]:
# CELL 3 — Install diffusers + transformers (run after restarting session)
# Should print: 2.4.1+cu121 and 0.29.2
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pip", "install",
    "diffusers==0.31.0",
    "transformers==4.47.0",
    "accelerate",
    "safetensors",
    "peft==0.14.0"
], capture_output=True, text=True)

print(result.stdout[-3000:])
print(result.stderr[-2000:])

import torch, diffusers
print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# CELL 4 — Install Kohya-ss sd-scripts (run once per session)
# Takes 3-5 minutes. Wait for 'Kohya-ss ready' before continuing.

import os
os.chdir("/content")
os.system("git clone https://github.com/kohya-ss/sd-scripts.git")
os.chdir("/content/sd-scripts")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q lion-pytorch prodigyopt")

print("Kohya-ss ready")

In [ ]:
# CELL 5 — Verify training directory (dataset already prepared, this just confirms counts)
# Should show 110 images and 110 captions. If not, something is wrong with prod_train.

from pathlib import Path

TRAIN_IMG_DIR   = Path(f"{TRAIN_DATA_DIR}/10_BushraStyle")
OUTPUT_DIR_LORA = Path(OUTPUT_DIR_LORA)
LOG_DIR         = Path(LOG_DIR)

OUTPUT_DIR_LORA.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

imgs = list(TRAIN_IMG_DIR.glob("*.jpg"))
txts = list(TRAIN_IMG_DIR.glob("*.txt"))
print(f"Training directory: {len(imgs)} images, {len(txts)} captions")
print(f"Output dir: {OUTPUT_DIR_LORA}")
assert len(imgs) > 0, "No images found! Check that prod_train/img/10_BushraStyle exists in Drive."
assert len(imgs) == len(txts), f"Mismatch: {len(imgs)} images but {len(txts)} captions!"
print("All good — ready to train")

In [ ]:
# CELL 6 — Train the LoRA (RESUMING from step 4000)
import os


TRAIN_CMD = (
    f'python /content/sd-scripts/train_network.py'
    f' --pretrained_model_name_or_path="{BASE_MODEL}"'
    f' --train_data_dir="{TRAIN_DATA_DIR}"'
    f' --output_dir="{OUTPUT_DIR_LORA}"'
    f' --logging_dir="{LOG_DIR}"'
    f' --output_name="BushraStyle_prod_v1"'
    f' --save_model_as=safetensors'
    f' --network_module=networks.lora'
    f' --network_dim=32'
    f' --network_alpha=16'
    f' --resolution=768,768'
    f' --train_batch_size=1'
    f' --max_train_steps=8000'
    f' --save_every_n_steps=500'
    f' --learning_rate=8e-5'
    f' --unet_lr=8e-5'
    f' --text_encoder_lr=4e-5'
    f' --lr_scheduler=cosine_with_restarts'
    f' --lr_warmup_steps=400'
    f' --optimizer_type=AdamW8bit'
    f' --mixed_precision=fp16'
    f' --gradient_checkpointing'
    f' --gradient_accumulation_steps=2'
    f' --cache_latents'
    f' --caption_extension=.txt'
    f' --shuffle_caption'
    f' --clip_skip=2'
    f' --noise_offset=0.05'
    f' --min_snr_gamma=5'
    f' --bucket_no_upscale'
    f' --min_bucket_reso=768'
    f' --max_bucket_reso=768'
    f' --seed=42'
    f' --network_weights="{CHECKPOINT}"'
)

print("Resuming LoRA training from step 4000...")
print("Remaining: ~4000 steps, estimated ~4 hours on T4")
print("Checkpoints saved every 500 steps to Drive\n")
os.system(TRAIN_CMD)

In [ ]:
# CELL 7 — Load ControlNet pipeline
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from pathlib import Path
import torch

OUTPUT_DIR_LORA = Path(OUTPUT_DIR_LORA)

print("Loading ControlNet models...")
controlnet_canny = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)
controlnet_depth = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=[controlnet_canny, controlnet_depth],
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe_cn.scheduler = UniPCMultistepScheduler.from_config(pipe_cn.scheduler.config)
pipe_cn.load_lora_weights(str(OUTPUT_DIR_LORA), weight_name="BushraStyle_prod_v1-step00004000.safetensors")

print("ControlNet pipeline ready")

In [ ]:
# CELL 8 — Generate one final work (run once per artwork)
# Edit SOURCE_PHOTO_PATH, OUTPUT_FILENAME, and PROMPT for each of the 8 works.
# Outputs: finals/work_08.png + comparison strip + control maps

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

import cv2
import numpy as np
from PIL import Image
from transformers import DPTImageProcessor, DPTForDepthEstimation
import torch
from pathlib import Path

# ----- EDIT THESE FOR EACH WORK -----
SOURCE_PHOTO_PATH = f"{DRIVE_ROOT}/raw_input/DSCF2385.JPG"
OUTPUT_FILENAME   = "work_13"
PROMPT = (
    "BushraStyle, traditional Arabian antique shop interior, shelves lined with brass dallah "
    "coffee pots and ornate wooden chests, narrow corridor of treasures, warm amber lantern light, "
    "dark intimate atmosphere, 1950s Arabia, film grain, analog tones, photographic"
)
RESOLUTION   = 768
CANNY_WEIGHT = 0.6
DEPTH_WEIGHT = 0.4
LORA_SCALE   = 0.85
SEED         = 42

FINALS_DIR = Path(f"{DRIVE_ROOT}/finals")
FINALS_DIR.mkdir(exist_ok=True)

# Load and crop source photo
img = Image.open(SOURCE_PHOTO_PATH).convert("RGB")
w, h = img.size
scale = RESOLUTION / min(w, h)
img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
w2, h2 = img.size
left = (w2 - RESOLUTION) // 2
top  = (h2 - RESOLUTION) // 2
img  = img.crop((left, top, left + RESOLUTION, top + RESOLUTION))
img_np = np.array(img)

# Extract Canny edges
print("Extracting Canny edges...")
gray     = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
blurred  = cv2.GaussianBlur(gray, (5, 5), 0)
canny_np = cv2.Canny(blurred, threshold1=80, threshold2=160)
kernel   = np.ones((2, 2), np.uint8)
canny_np = cv2.dilate(canny_np, kernel, iterations=1)
canny_img = Image.fromarray(np.stack([canny_np]*3, axis=-1))

# Extract depth map
print("Extracting depth map (MiDaS)...")
depth_processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")
depth_model     = DPTForDepthEstimation.from_pretrained(
    "Intel/dpt-large", torch_dtype=torch.float16
).to("cuda")
depth_inputs = depth_processor(images=img, return_tensors="pt")
depth_inputs = {k: v.to("cuda", torch.float16) for k, v in depth_inputs.items()}
with torch.no_grad():
    depth_output = depth_model(**depth_inputs).predicted_depth
depth_np = depth_output.squeeze().cpu().numpy()
depth_np = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min()) * 255
depth_np = depth_np.astype(np.uint8)
depth_np = cv2.resize(depth_np, (RESOLUTION, RESOLUTION))
depth_img = Image.fromarray(np.stack([depth_np]*3, axis=-1))

# Save control maps for documentation
canny_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_canny.png")
depth_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_depth.png")
print("Control maps saved")

# Generate
print("Generating final image...")
negative_prompt = (
    "digital art, 3d render, oversaturated, neon, modern building, HDR, "
    "cartoon, anime, blurry, ugly, watermark, plastic, chrome, glass facade, "
    "people, figures, humans, persons, man, woman, crowd, text, writing, signs, "
    "arabic text, letters, words, daylight, bright, white light, electric lights, LED, "
    "bright, daylight, daytime, sunlight, overexposed, grey, flat lighting, "
)
generator = torch.Generator("cuda").manual_seed(SEED)
result = pipe_cn(
    prompt=PROMPT,
    image=[canny_img, depth_img],
    controlnet_conditioning_scale=[CANNY_WEIGHT, DEPTH_WEIGHT],
    negative_prompt=negative_prompt,
    num_inference_steps=35,
    guidance_scale=8.0,
    width=RESOLUTION,
    height=RESOLUTION,
    generator=generator,
    cross_attention_kwargs={"scale": LORA_SCALE},
).images[0]

result.save(FINALS_DIR / f"{OUTPUT_FILENAME}.png")

# Save comparison strip: original | canny | depth | generated
compare = Image.new("RGB", (RESOLUTION * 4, RESOLUTION))
compare.paste(img,       (0, 0))
compare.paste(canny_img, (RESOLUTION, 0))
compare.paste(depth_img, (RESOLUTION*2, 0))
compare.paste(result,    (RESOLUTION*3, 0))
compare.save(FINALS_DIR / f"{OUTPUT_FILENAME}_comparison.png")

print(f"\nDone.")
print(f"Final image:  Drive/LostArchive/finals/{OUTPUT_FILENAME}.png")
print(f"Comparison:   Drive/LostArchive/finals/{OUTPUT_FILENAME}_comparison.png")
print(f"Control maps: Drive/LostArchive/finals/{OUTPUT_FILENAME}_canny.png + _depth.png")

In [ ]:
# CELL 8 — Generate one final work (run once per artwork)
# Outputs: finals/work_03.png + comparison strip + control maps

import cv2
import numpy as np
from PIL import Image
from transformers import DPTImageProcessor, DPTForDepthEstimation
import torch
from pathlib import Path

# ----- EDIT THESE FOR EACH WORK -----
SOURCE_PHOTO_PATH = f"{DRIVE_ROOT}/raw_input/DSCF2394.JPG"
OUTPUT_FILENAME   = "work_03"
PROMPT = (
    "BushraStyle, ancient Saudi street scene, weathered stone walls, "
    "warm afternoon light, 1950s Arabia, film grain, analog tones, photographic"
)
# ------------------------------------

RESOLUTION   = 768
CANNY_WEIGHT = 0.6
DEPTH_WEIGHT = 0.4
LORA_SCALE   = 0.85
SEED         = 42

FINALS_DIR = Path(f"{DRIVE_ROOT}/finals")
FINALS_DIR.mkdir(exist_ok=True)

img = Image.open(SOURCE_PHOTO_PATH).convert("RGB")
w, h = img.size
scale = RESOLUTION / min(w, h)
img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
w2, h2 = img.size
left = (w2 - RESOLUTION) // 2
top  = (h2 - RESOLUTION) // 2
img  = img.crop((left, top, left + RESOLUTION, top + RESOLUTION))
img_np = np.array(img)

print("Extracting Canny edges...")
gray     = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
blurred  = cv2.GaussianBlur(gray, (5, 5), 0)
canny_np = cv2.Canny(blurred, threshold1=80, threshold2=160)
kernel   = np.ones((2, 2), np.uint8)
canny_np = cv2.dilate(canny_np, kernel, iterations=1)
canny_img = Image.fromarray(np.stack([canny_np]*3, axis=-1))

print("Extracting depth map (MiDaS)...")
depth_processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")
depth_model     = DPTForDepthEstimation.from_pretrained(
    "Intel/dpt-large", torch_dtype=torch.float16
).to("cuda")
depth_inputs = depth_processor(images=img, return_tensors="pt")
depth_inputs = {k: v.to("cuda", torch.float16) for k, v in depth_inputs.items()}
with torch.no_grad():
    depth_output = depth_model(**depth_inputs).predicted_depth
depth_np = depth_output.squeeze().cpu().numpy()
depth_np = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min()) * 255
depth_np = depth_np.astype(np.uint8)
depth_np = cv2.resize(depth_np, (RESOLUTION, RESOLUTION))
depth_img = Image.fromarray(np.stack([depth_np]*3, axis=-1))

canny_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_canny.png")
depth_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_depth.png")
print("Control maps saved")

print("Generating final image...")
negative_prompt = (
    "digital art, 3d render, oversaturated, neon, modern building, HDR, "
    "cartoon, anime, blurry, ugly, watermark, plastic, chrome, glass facade"
)
generator = torch.Generator("cuda").manual_seed(SEED)
result = pipe_cn(
    prompt=PROMPT,
    image=[canny_img, depth_img],
    controlnet_conditioning_scale=[CANNY_WEIGHT, DEPTH_WEIGHT],
    negative_prompt=negative_prompt,
    num_inference_steps=35,
    guidance_scale=8.0,
    width=RESOLUTION,
    height=RESOLUTION,
    generator=generator,
    cross_attention_kwargs={"scale": LORA_SCALE},
).images[0]

result.save(FINALS_DIR / f"{OUTPUT_FILENAME}.png")

compare = Image.new("RGB", (RESOLUTION * 4, RESOLUTION))
compare.paste(img,       (0, 0))
compare.paste(canny_img, (RESOLUTION, 0))
compare.paste(depth_img, (RESOLUTION*2, 0))
compare.paste(result,    (RESOLUTION*3, 0))
compare.save(FINALS_DIR / f"{OUTPUT_FILENAME}_comparison.png")

print(f"\nDone.")
print(f"Final image:  Drive/LostArchive/finals/{OUTPUT_FILENAME}.png")
print(f"Comparison:   Drive/LostArchive/finals/{OUTPUT_FILENAME}_comparison.png")
print(f"Control maps: Drive/LostArchive/finals/{OUTPUT_FILENAME}_canny.png + _depth.png")

In [ ]:
# CELL 8 — Generate one final work (run once per artwork)
# Outputs: finals/work_03.png + comparison strip + control maps

import cv2
import numpy as np
from PIL import Image
from transformers import DPTImageProcessor, DPTForDepthEstimation
import torch
from pathlib import Path

# ----- EDIT THESE FOR EACH WORK -----
SOURCE_PHOTO_PATH = f"{DRIVE_ROOT}/raw_input/DSCF2421.jpg"
OUTPUT_FILENAME   = "work_03"
PROMPT = (
    "BushraStyle, old Riyadh souk alley at night, hanging lanterns casting warm golden glow, "
    "empty market corridor, Persian rugs on floor, merchant stalls, atmospheric haze, "
    "1950s Saudi Arabia, film grain, analog tones, cinematic, photographic"
)
# ------------------------------------

RESOLUTION   = 768
CANNY_WEIGHT = 0.6
DEPTH_WEIGHT = 0.4
LORA_SCALE   = 0.85
SEED         = 42

FINALS_DIR = Path(f"{DRIVE_ROOT}/finals")
FINALS_DIR.mkdir(exist_ok=True)

img = Image.open(SOURCE_PHOTO_PATH).convert("RGB")
w, h = img.size
scale = RESOLUTION / min(w, h)
img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
w2, h2 = img.size
left = (w2 - RESOLUTION) // 2
top  = (h2 - RESOLUTION) // 2
img  = img.crop((left, top, left + RESOLUTION, top + RESOLUTION))
img_np = np.array(img)

print("Extracting Canny edges...")
gray     = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
blurred  = cv2.GaussianBlur(gray, (5, 5), 0)
canny_np = cv2.Canny(blurred, threshold1=80, threshold2=160)
kernel   = np.ones((2, 2), np.uint8)
canny_np = cv2.dilate(canny_np, kernel, iterations=1)
canny_img = Image.fromarray(np.stack([canny_np]*3, axis=-1))

print("Extracting depth map (MiDaS)...")
depth_processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")
depth_model     = DPTForDepthEstimation.from_pretrained(
    "Intel/dpt-large", torch_dtype=torch.float16
).to("cuda")
depth_inputs = depth_processor(images=img, return_tensors="pt")
depth_inputs = {k: v.to("cuda", torch.float16) for k, v in depth_inputs.items()}
with torch.no_grad():
    depth_output = depth_model(**depth_inputs).predicted_depth
depth_np = depth_output.squeeze().cpu().numpy()
depth_np = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min()) * 255
depth_np = depth_np.astype(np.uint8)
depth_np = cv2.resize(depth_np, (RESOLUTION, RESOLUTION))
depth_img = Image.fromarray(np.stack([depth_np]*3, axis=-1))

canny_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_canny.png")
depth_img.save(FINALS_DIR / f"{OUTPUT_FILENAME}_depth.png")
print("Control maps saved")

print("Generating final image...")
negative_prompt = (
    "digital art, 3d render, oversaturated, neon, modern building, HDR, "
    "cartoon, anime, blurry, ugly, watermark, plastic, chrome, glass facade, "
    "QR code, signage, latin text, electric lights, LED, "
    "people, figures, humans, persons, man, woman, crowd, "
    "text, writing, signs, arabic text, letters, words"
)
generator = torch.Generator("cuda").manual_seed(SEED)
result = pipe_cn(
    prompt=PROMPT,
    image=[canny_img, depth_img],
    controlnet_conditioning_scale=[CANNY_WEIGHT, DEPTH_WEIGHT],
    negative_prompt=negative_prompt,
    num_inference_steps=35,
    guidance_scale=8.0,
    width=RESOLUTION,
    height=RESOLUTION,
    generator=generator,
    cross_attention_kwargs={"scale": LORA_SCALE},
).images[0]

result.save(FINALS_DIR / f"{OUTPUT_FILENAME}.png")

compare = Image.new("RGB", (RESOLUTION * 4, RESOLUTION))
compare.paste(img,       (0, 0))
compare.paste(canny_img, (RESOLUTION, 0))
compare.paste(depth_img, (RESOLUTION*2, 0))
compare.paste(result,    (RESOLUTION*3, 0))
compare.save(FINALS_DIR / f"{OUTPUT_FILENAME}_comparison.png")

print(f"\nDone.")
print(f"Final image:  Drive/LostArchive/finals/{OUTPUT_FILENAME}.png")
print(f"Comparison:   Drive/LostArchive/finals/{OUTPUT_FILENAME}_comparison.png")
print(f"Control maps: Drive/LostArchive/finals/{OUTPUT_FILENAME}_canny.png + _depth.png")

In [ ]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path
from spandrel import ImageModelDescriptor, ModelLoader

# Download RealESRGAN weights if not already there
os.makedirs("/content/weights", exist_ok=True)
os.system(
    "wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth "
    "-P /content/weights/"
)

# Load model via spandrel
model = ModelLoader().load_from_file("/content/weights/RealESRGAN_x4plus.pth")
assert isinstance(model, ImageModelDescriptor)
model.cuda().eval()

FINALS_DIR   = Path(f"{DRIVE_ROOT}/finals")
UPSCALED_DIR = Path(f"{DRIVE_ROOT}/finals_upscaled")
UPSCALED_DIR.mkdir(exist_ok=True)

for img_file in sorted(FINALS_DIR.glob("*.png")):
    if any(x in img_file.name for x in ["_comparison", "_canny", "_depth"]):
        continue
    print(f"Upscaling {img_file.name}...")
    img = cv2.imread(str(img_file), cv2.IMREAD_COLOR)
    # Convert to tensor
    img_t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float() / 255.0
    img_t = img_t.cuda()
    with torch.no_grad():
        output = model(img_t)
    output = output.squeeze(0).permute(1,2,0).cpu().numpy()
    output = (output * 255).clip(0, 255).astype(np.uint8)
    out_path = UPSCALED_DIR / f"{img_file.stem}_4x.png"
    cv2.imwrite(str(out_path), output)
    h, w = output.shape[:2]
    print(f"Done: {w}x{h}px -> {out_path.name}")

print("\nAll finals upscaled. Check Drive > LostArchive/finals_upscaled/")

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pip", "install",
    "spandrel",
    "safetensors"
], capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-1000:])

In [ ]:
from pathlib import Path

basicsr_file = Path("/usr/local/lib/python3.13/dist-packages/basicsr/data/degradations.py")
content = basicsr_file.read_text()
content = content.replace(
    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
    "from torchvision.transforms.functional import rgb_to_grayscale"
)
basicsr_file.write_text(content)
print("basicsr patched")